# Lección 1.2 — Vocabulario esencial: dato, característica, etiqueta, modelo, predicción, parámetro e hiperparámetro

> **Módulo 1 · ¿Qué es aprender para una máquina?** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 60–75 min (lección con ejercicios intercalados ≈ 40 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 1.1 (programación tradicional vs. aprendizaje a partir de ejemplos).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Usar con precisión las palabras **ejemplo, característica (*feature*), etiqueta (*label/target*), modelo, entrenamiento y predicción**.
- Identificar `X` e `y` en cualquier tabla de datos.
- Distinguir **parámetros** (lo que el modelo aprende) de **hiperparámetros** (lo que tú eliges antes de entrenar).
- Detectar una trampa clásica: una característica que "contiene" la respuesta.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números o palabras.

## ▶️ Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

print("✅ Todo listo")

---
## 1. 📖 La intuición: el agente inmobiliario con experiencia

Una agente inmobiliaria con 20 años de oficio ve un departamento y dice: *"Este vale unos 250 mil"*. ¿Cómo lo sabe?

- Ha visto **cientos de departamentos** vendidos → los **datos** (cada uno es un **ejemplo**).
- De cada uno se fija en ciertas cosas: metros, habitaciones, cercanía al metro, antigüedad → las **características**.
- Sabe **en cuánto se vendió** cada uno → la **etiqueta** (la respuesta correcta).
- Con los años desarrolló un "olfato" → el **modelo**.
- Formar ese olfato le llevó tiempo → el **entrenamiento**.
- Ante un departamento nuevo, da una cifra → la **predicción**.

Todo el vocabulario del ML está en esta historia.

### ✋ Ejercicio 1 — Traduce la historia
Un médico con experiencia mira una radiografía y dice "esto parece neumonía". Identifica en esta situación: los **ejemplos**, las **características**, la **etiqueta**, el **modelo** y la **predicción**.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Ejemplos:** las miles de radiografías que vio durante su carrera.
- **Características:** lo que observa en cada una (manchas, su forma, su ubicación, densidad…). En un modelo de imágenes, serían los píxeles.
- **Etiqueta:** el diagnóstico confirmado de cada paciente (neumonía o no).
- **Modelo:** su "ojo clínico", formado con la experiencia.
- **Predicción:** "esto parece neumonía" para la radiografía nueva.
</details>

---
## 2. 📖 Ejemplos, características y etiqueta

Vamos a ponerle datos a la historia: el historial de **200 departamentos vendidos**.

In [ ]:
rng = np.random.default_rng(42)
n = 200
deptos = pd.DataFrame({
    "metros":       rng.integers(30, 150, n),
    "distancia_metro_km": np.round(rng.uniform(0.1, 5, n), 1),
    "antiguedad":   rng.integers(0, 50, n),
})
deptos["habitaciones"] = np.clip(deptos.metros // 30 + rng.integers(-1, 2, n), 1, 5)
# Precio de venta en miles de $ (la regla real, con ruido, que el modelo NO conoce)
deptos["precio"] = np.round(40 + 2.0 * deptos.metros + 8 * deptos.habitaciones
                            - 12 * deptos.distancia_metro_km - 0.9 * deptos.antiguedad
                            + rng.normal(0, 15, n)).astype(int)
deptos.head()

| Término | Qué es | En la tabla |
|---|---|---|
| **Ejemplo** (*sample*, *instance*, observación) | Un caso concreto | Una **fila**: un departamento |
| **Característica** (*feature*, variable, atributo) | Un dato que describe al ejemplo y que usamos para predecir | Las **columnas de entrada**: `metros`, `habitaciones`… |
| **Etiqueta** (*label*, *target*, objetivo) | Lo que queremos predecir | La columna `precio` |

Por convención, en ML a la tabla de características se le llama **`X`** (mayúscula, porque es una tabla) y a la etiqueta **`y`** (minúscula, porque es una sola columna).

### 🔮 Ejercicio 2 — Cuenta
Sin ejecutar nada: ¿cuántos **ejemplos** y cuántas **características** tiene `X`? ¿Cuántos valores tiene `y`?

✍️ *Mis predicciones:*

In [ ]:
X = deptos[["metros", "habitaciones", "distancia_metro_km", "antiguedad"]]   # características
y = deptos["precio"]                                                           # etiqueta

print("X:", X.shape, "→ (ejemplos, características)")
print("y:", y.shape)

<details><summary><b>👀 Ver respuesta</b></summary>

`X` tiene forma **(200, 4)**: 200 ejemplos y 4 características. `y` tiene **200** valores: una etiqueta por ejemplo. Regla de oro: **`X` e `y` siempre tienen el mismo número de filas**.
</details>

---
## 3. 📖 Modelo, entrenamiento y predicción

- El **modelo** es la "máquina de predecir". Antes de entrenar es una caja vacía.
- **Entrenar** (*train*, *fit*) es mostrarle los ejemplos (`X`) con sus respuestas (`y`) para que ajuste sus reglas.
- **Predecir** (*predict*, inferencia) es pedirle una respuesta para un ejemplo **nuevo**, del que no conocemos la etiqueta.

En scikit-learn, esas tres ideas son tres líneas que verás en **todo** el curso:

```python
modelo = UnModelo()        # 1. crear el modelo (vacío)
modelo.fit(X, y)           # 2. entrenar: aprende de ejemplos + respuestas
modelo.predict(X_nuevo)    # 3. predecir: responde para casos nuevos
```

In [ ]:
modelo = LinearRegression()        # 1. crear
modelo.fit(X, y)                   # 2. entrenar

depto_nuevo = pd.DataFrame({"metros": [80], "habitaciones": [2], "distancia_metro_km": [1.0], "antiguedad": [10]})
print(f"Precio estimado: {modelo.predict(depto_nuevo)[0]:.0f} mil $")   # 3. predecir

### ✏️ Ejercicio 3 — Tú eres el cliente
Cambia los valores de `depto_nuevo` para responder (🔮 apuesta antes de cada ejecución):
1. ¿Sube o baja el precio si el departamento está a **4 km** del metro en lugar de 1 km? ¿Cuánto?
2. ¿Y si tiene **40 años** de antigüedad en lugar de 10?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con los valores originales el modelo estima **≈ 194 mil $**.
1. A 4 km del metro → **≈ 156 mil $**: baja unos **39 mil** (casi 13 mil por cada km extra).
2. Con 40 años → **≈ 171 mil $**: baja unos **24 mil** (unos 0.8 mil por año).

¿De dónde salen esos "por cada km" y "por año"? De los **parámetros** del modelo. Siguiente sección.
</details>

---
## 4. 📖 Parámetros: lo que el modelo aprende

Los **parámetros** son los números internos que el modelo **ajusta durante el entrenamiento**. Nadie los escribe a mano: salen de los datos. (En la Lección 1.1, el 1.8 y el 32 de la fórmula °C → °F eran parámetros.)

Una regresión lineal aprende **un número por característica** (cuánto suma o resta cada una) más un término fijo:

In [ ]:
parametros = pd.Series(modelo.coef_, index=X.columns).round(2)
print(parametros)
print(f"término fijo: {modelo.intercept_:.2f}")

### ✋ Ejercicio 4 — Lee los parámetros
1. Según el modelo, ¿cuánto suma al precio **cada metro cuadrado** extra?
2. ¿Qué característica **resta** más por cada unidad?
3. La regla real con la que se generaron los datos era: *40 + 2.0·metros + 8·habitaciones − 12·distancia − 0.9·antigüedad (+ ruido)*. ¿Se parecen los parámetros aprendidos?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Unos **1.99 mil $** por m² (la regla real: 2.0).
2. La **distancia al metro**: unos **−12.93** por cada km (real: −12).
3. Sí, bastante: metros 1.99 (real 2.0), habitaciones 6.56 (real 8), distancia −12.93 (real −12), antigüedad −0.79 (real −0.9), término fijo 43.04 (real 40). El que más se aleja es **habitaciones**: como va muy ligada a los metros (más metros → más habitaciones), al modelo le cuesta separar el efecto de cada una. Con datos ruidosos, los parámetros aprendidos son **estimaciones**, no la verdad exacta.
</details>

---
## 5. 📖 Hiperparámetros: lo que eliges tú antes de entrenar

Los **hiperparámetros** son las "perillas de configuración" del modelo: **tú** las fijas **antes** de entrenar, y cambian **cómo** aprende.

Analogía: un estudiante prepara un examen.
- Lo que **aprende** (fórmulas, fechas, conceptos) → **parámetros**.
- **Cómo** decide estudiar (cuántas horas, con resúmenes o con ejercicios, solo o en grupo) → **hiperparámetros**.

Ejemplo: el modelo **KNN** (*K vecinos más cercanos*, Lección 5.3) estima el precio de un departamento promediando el precio de los **k** departamentos más parecidos. Ese **k** es un hiperparámetro: lo decides tú.

### 🔮 Ejercicio 5 — Gira la perilla
Vamos a estimar el precio del mismo departamento nuevo con **k = 1, 5, 50 y 200** vecinos.
1. ¿Crees que la estimación cambiará al cambiar k, si los datos son los mismos?
2. Con **k = 200** el modelo consulta **todos** los departamentos. ¿Qué número predecirá?

✍️ *Mi predicción:*

In [ ]:
print(f"Precio promedio de los 200 departamentos: {y.mean():.0f} mil $\n")
for k in [1, 5, 50, 200]:        # ✏️ prueba otros valores (máximo 200)
    knn = KNeighborsRegressor(n_neighbors=k)      # ← el hiperparámetro se fija AL CREAR el modelo
    knn.fit(X, y)                                 # ← aquí aprende (KNN, en realidad, "memoriza" los ejemplos)
    print(f"k = {k:>3} → precio estimado: {knn.predict(depto_nuevo)[0]:.0f} mil $")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Sí cambia:** k = 1 → **177**, k = 5 → **183**, k = 50 → **172** mil $. Con k = 1 la estimación depende de **un único** vecino (si ese departamento tuvo un precio raro, la estimación también); con k = 50 se mezclan departamentos cada vez menos parecidos.
2. Con **k = 200** predice **188**: exactamente el **precio promedio** de todos los departamentos. Promedia todo, así que **ignora** por completo las características del departamento nuevo: daría 188 para un estudio de 30 m² y para un piso de 150 m².

Fíjate: **los datos son los mismos**; solo cambiaste una perilla y la predicción cambió. Elegir bien los hiperparámetros es una parte central del ML (Módulo 7).
</details>

### 📖 Parámetro vs. hiperparámetro, lado a lado

| | **Parámetro** | **Hiperparámetro** |
|---|---|---|
| ¿Quién lo decide? | El **modelo**, al entrenar | **Tú**, antes de entrenar |
| ¿Cuándo? | Durante `fit` | Al crear el modelo: `Modelo(hiperparámetro=...)` |
| ¿Cómo se elige? | Lo calcula el algoritmo a partir de los datos | Probando y validando (Lección 4.6, Módulo 7) |
| Ejemplo en regresión lineal | Los coeficientes (1.99 por m²…) | — (casi no tiene) |
| Ejemplo en KNN | — (memoriza los datos) | `n_neighbors` (k) |
| Ejemplo en árbol de decisión | Las preguntas y los cortes ("¿metros ≤ 75?") | `max_depth` (Lección 5.4) |
| Ejemplo en red neuronal | Los pesos (las 2,778 perillas de la Lección 0.2) | Número de capas, tasa de aprendizaje |

### ✋ Ejercicio 6 — ¿Parámetro o hiperparámetro?
1. El número máximo de preguntas que puede hacer un árbol de decisión.
2. El valor "1.8" que aprendió el modelo de °C → °F.
3. El número de capas de una red neuronal.
4. El umbral "humedad ≤ 34.5" que encontró el árbol del jardinero (Lección 0.2).
5. Cuántas veces (épocas) recorre una red neuronal los datos al entrenar.

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Hiperparámetro** (`max_depth`: lo eliges tú). 2. **Parámetro** (lo aprendió). 3. **Hiperparámetro**. 4. **Parámetro** (el árbol lo descubrió en los datos). 5. **Hiperparámetro** (Lección 3.4).

Truco: **si lo puedes escribir antes de ver los datos, es un hiperparámetro**.
</details>

---
## 6. 🚨 Errores comunes y señales de alerta

| 🚨 Error | Qué pasa | Cómo evitarlo |
|---|---|---|
| Meter la etiqueta (o algo que la "contiene") entre las características | El modelo parece perfecto, pero es trampa: con casos nuevos no tendrás ese dato | Pregúntate: **¿tendré esta columna en el momento de predecir?** (Lección 2.6) |
| Usar identificadores (ID, nombre) como característica | El modelo memoriza nombres, no patrones | Quita las columnas que solo identifican |
| `X` e `y` con distinto número de filas | Error al entrenar | Sácalos de la **misma** tabla |
| Confundir parámetros con hiperparámetros | Intentar "fijar a mano" lo que el modelo debe aprender, o esperar que aprenda lo que debes elegir | Regla: hiperparámetro = antes de ver los datos |

### 🔮 Ejercicio 7 — El modelo "perfecto"
Un compañero añade la columna `precio_por_m2` (= precio / metros) a las características y obtiene un modelo casi perfecto. Antes de ejecutar: ¿te parece una buena noticia?

✍️ *Mi opinión:*

In [ ]:
X_trampa = X.assign(precio_por_m2=deptos.precio / deptos.metros)
modelo_trampa = LinearRegression().fit(X_trampa, y)
modelo_honesto = LinearRegression().fit(X, y)

print(f"Error medio del modelo honesto: {np.mean(abs(modelo_honesto.predict(X) - y)):.1f} mil $")
print(f"Error medio del modelo 'trampa': {np.mean(abs(modelo_trampa.predict(X_trampa) - y)):.1f} mil $")

<details><summary><b>👀 Ver respuesta</b></summary>

El modelo "trampa" se equivoca bastante menos (7.5 mil $ contra 12.0 mil $)… pero es **inútil**: para calcular `precio_por_m2` necesitas **el precio**, que es justo lo que quieres predecir. Cuando llegue un departamento nuevo, esa columna no existirá.

Es una **fuga de datos** (*data leakage*): información de la respuesta que se cuela en las características. Es el error silencioso más caro del ML (Lección 2.6). Pregunta salvavidas: **¿tendré esta columna en el momento de predecir?**
</details>

---
## 7. 📋 Mini-glosario de la lección

| Español | Inglés | En una línea |
|---|---|---|
| Ejemplo / observación | *sample, instance* | Una fila de la tabla: un caso |
| Característica / variable | *feature* | Una columna de entrada que describe al ejemplo (`X`) |
| Etiqueta / objetivo | *label, target* | Lo que queremos predecir (`y`) |
| Modelo | *model* | La función que convierte características en predicciones |
| Entrenar | *train, fit* | Ajustar el modelo con ejemplos y sus respuestas |
| Predecir / inferencia | *predict, inference* | Usar el modelo con ejemplos nuevos |
| Parámetro | *parameter* | Número que el modelo **aprende** al entrenar |
| Hiperparámetro | *hyperparameter* | Configuración que **tú eliges** antes de entrenar |
| Fuga de datos | *data leakage* | Información de la respuesta que se cuela en las características |

## 8. 🧠 Resumen en 5 puntos
1. Los datos son una tabla: cada **fila** es un **ejemplo** y cada **columna de entrada**, una **característica** (`X`).
2. La **etiqueta** (`y`) es lo que queremos predecir; en el entrenamiento la conocemos, en la predicción no.
3. Todo modelo de scikit-learn sigue el patrón: **crear → `fit(X, y)` → `predict(X_nuevo)`**.
4. **Parámetros** = lo que el modelo aprende. **Hiperparámetros** = lo que tú eliges antes.
5. Pregunta salvavidas para cada característica: **¿la tendré en el momento de predecir?**

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** En una tabla de 500 clientes con 8 columnas de datos y una columna "¿canceló?", ¿qué forma tiene `X`?
- a) (8, 500) · b) (500, 8) · c) (500, 9) · d) (500, 1)

**2.** ¿Qué hace `modelo.fit(X, y)`?
- a) Predice · b) Entrena · c) Borra el modelo · d) Divide los datos

**3.** En `KNeighborsClassifier(n_neighbors=7)`, el 7 es…
- a) Un parámetro · b) Un hiperparámetro · c) Una etiqueta · d) Una característica

**4.** Quieres predecir cuánto gastará un cliente el próximo mes. ¿Cuál de estas columnas es **peligrosa** como característica?
- a) Gasto del mes pasado · b) Edad · c) Gasto total del próximo mes según la factura · d) Número de visitas este mes

**5.** ¿Cuál de estas frases es correcta?
- a) Los hiperparámetros se aprenden de los datos · b) Los parámetros se fijan antes de entrenar · c) Los parámetros se aprenden de los datos al entrenar · d) Son lo mismo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** (500 ejemplos, 8 características; la columna "¿canceló?" es `y`) · 2. **b)** · 3. **b)** · 4. **c)** Es la propia respuesta: fuga de datos · 5. **c)**
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Spotify quiere predecir si te gustará una canción nueva. Propón: qué sería un **ejemplo**, **tres características** y la **etiqueta**. ¿De dónde sacaría Spotify esa etiqueta si nadie le dice explícitamente "me gustó"?

**B.** ¿Por qué crees que los hiperparámetros no se pueden aprender con el mismo `fit` que los parámetros? Piensa en el ejemplo de KNN con k = 1: ¿qué k elegiría el modelo si solo quisiera acertar en los datos que ya vio?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** Ejemplo: un par (usuario, canción). Características: género de la canción, su tempo, cuántas veces escuchaste al artista, la hora del día, si la canción es popular entre usuarios parecidos a ti. Etiqueta: "le gustó" o "no". Como nadie lo dice explícitamente, Spotify la **deduce** de tu conducta: si la escuchaste completa, la guardaste o la repetiste (le gustó), o si la saltaste en los primeros segundos (no le gustó).

**B.** Con k = 1, cada departamento de entrenamiento tiene como vecino más cercano **a sí mismo**: el error en los datos vistos sería cero. Si el modelo eligiera su propio k mirando solo esos datos, elegiría siempre k = 1… y memorizaría. Por eso los hiperparámetros se eligen **con datos que el modelo no usó para entrenar** (validación, Lección 4.6).
</details>

### 9.3 🧩 Práctica integradora: el botánico automático 🌸

El conjunto de datos **Iris** es el "hola, mundo" del ML: 150 flores de 3 especies, con 4 medidas de cada una. Vas a recorrer **todo el vocabulario** de la lección en 5 pasos. Revisa cada respuesta antes de seguir.

**Paso 1 ▶️ — Conoce los datos**

In [ ]:
iris = load_iris(as_frame=True)
flores = iris.frame.copy()
flores["especie"] = flores["target"].map(dict(enumerate(iris.target_names)))
flores = flores.drop(columns="target")
flores.sample(5, random_state=1)

✋ ¿Cuántos ejemplos hay? ¿Cuáles son las características? ¿Cuál es la etiqueta? (Pista: `flores.shape`.)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**150 ejemplos** (flores). **4 características**: largo y ancho del sépalo y largo y ancho del pétalo (en cm). **Etiqueta:** `especie` (setosa, versicolor o virginica).
</details>

**Paso 2 ✏️ — Arma `X` e `y`.** Completa las dos líneas (cambia los `None`) y comprueba las formas.

In [ ]:
X_flores = None   # ✏️ todas las columnas menos "especie"   (pista: flores.drop(columns="especie"))
y_flores = None   # ✏️ solo la columna "especie"

print(X_flores.shape if X_flores is not None else "Completa X_flores")
print(y_flores.shape if y_flores is not None else "Completa y_flores")

<details><summary><b>👀 Ver respuesta</b></summary>

```python
X_flores = flores.drop(columns="especie")
y_flores = flores["especie"]
```
Formas: **(150, 4)** y **(150,)**.
</details>

**Paso 3 ▶️ — Separa un examen y entrena.** Guardamos 30 flores que el modelo **no verá** al entrenar, para examinarlo (lo estudiarás a fondo en la Lección 2.5). El hiperparámetro `N_VECINOS` lo eliges tú.

In [ ]:
X_flores = flores.drop(columns="especie")     # (por si no completaste el Paso 2)
y_flores = flores["especie"]
X_ent, X_exa, y_ent, y_exa = train_test_split(X_flores, y_flores, test_size=30, random_state=0, stratify=y_flores)

N_VECINOS = 5                                  # ✏️ hiperparámetro: prueba 1, 5, 15 y 100
botanico = KNeighborsClassifier(n_neighbors=N_VECINOS)
botanico.fit(X_ent, y_ent)
print(f"Con k = {N_VECINOS}: acierto en las 30 flores de examen = {botanico.score(X_exa, y_exa):.0%}")

🔮 Antes de probar cada valor de `N_VECINOS`, predice: ¿con cuál crees que le irá **peor**?

✍️ *Mis predicciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

- k = 1 → **97 %** · k = 5 → **100 %** · k = 15 → **100 %** · k = 100 → **67 %**.

Con k = 100 consulta a 100 de las 120 flores de entrenamiento: casi todas, de las tres especies. Su "opinión" se vuelve un promedio de todo y deja de distinguir. Con k = 1 depende de una sola flor vecina: más sensible a los casos raros. Los valores intermedios funcionan mejor. (Ojo: con un examen de solo 30 flores, la diferencia entre 97 % y 100 % es **una sola flor**.)
</details>

**Paso 4 ✏️ — Predice una flor nueva.** Encontraste una flor con sépalo de 6.0 × 3.0 cm y pétalo de 4.8 × 1.8 cm. ¿Qué especie es? (Deja `N_VECINOS = 5` en el paso anterior.)

In [ ]:
flor_nueva = pd.DataFrame([[6.0, 3.0, 4.8, 1.8]], columns=X_flores.columns)   # ✏️ prueba con tus propias medidas
print("Predicción:", botanico.predict(flor_nueva)[0])
print("Probabilidades:", dict(zip(map(str, botanico.classes_), botanico.predict_proba(flor_nueva)[0].round(2).tolist())))

<details><summary><b>👀 Ver respuesta</b></summary>

Con k = 5, predice **virginica** con probabilidad 0.8 (4 de sus 5 vecinos son virginica y 1 es versicolor). Es una flor "frontera": sus medidas están justo entre las dos especies. Que el modelo dé una **probabilidad** y no solo una respuesta es muy útil: te dice **qué tan seguro** está.
</details>

**Paso 5 ✋ — Repasa el vocabulario.** En esta práctica, ¿qué fue…?
1. un ejemplo · 2. una característica · 3. la etiqueta · 4. el modelo · 5. el entrenamiento · 6. una predicción · 7. un hiperparámetro

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Ejemplo:** cada flor (fila). 2. **Característica:** p. ej. el largo del pétalo. 3. **Etiqueta:** la especie. 4. **Modelo:** `botanico` (un KNN). 5. **Entrenamiento:** `botanico.fit(X_ent, y_ent)`. 6. **Predicción:** "virginica" para la flor nueva. 7. **Hiperparámetro:** `n_neighbors` (k).

¿Y los **parámetros**? KNN es un caso curioso: no aprende números, **guarda los ejemplos** y los consulta al predecir. Se le llama modelo "perezoso" (Lección 5.3).
</details>

---
## 📚 Para profundizar
- 📘 **Glosario de Google:** *Machine Learning Glossary* (developers.google.com/machine-learning/glossary), con definiciones breves de cientos de términos.
- 📘 **scikit-learn:** guía *Getting Started*, que muestra el patrón `fit` / `predict` en todos los modelos.
- 🎮 **Interactivo:** *R2D3 — A visual introduction to machine learning* (características, etiquetas y modelo explicados con casas de Nueva York y San Francisco).

---
⬅️ **Anterior:** [1.1 Programación tradicional vs. aprendizaje a partir de ejemplos](1.1-programacion-tradicional-vs-aprendizaje.ipynb) · ➡️ **Siguiente:** [1.3 Tipos de aprendizaje: supervisado, no supervisado, semi-supervisado, auto-supervisado y por refuerzo](1.3-tipos-de-aprendizaje.ipynb)